In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
!pip install transformers datasets soundfile librosa evaluate jiwer accelerate -q


In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "google/fleurs",
    "hi_in",
    split="train"
)

print(dataset)

Dataset({
    features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
    num_rows: 2120
})


In [ ]:
sample = dataset[0]

print("Transcription:", sample["transcription"])
print("Language:", sample["language"])
print("Audio sample rate:", sample["audio"]["sampling_rate"])
print("Audio array length:", len(sample["audio"]["array"]))

Transcription: राजनीतिज्ञों ने कहा कि उन्होंने निर्णायक मत को अनावश्यक रूप से निर्धारित करने के लिए अफ़गान संविधान में काफी अस्पष्टता पाई थी
Language: Hindi
Audio sample rate: 16000
Audio array length: 138240


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

print("Model loaded!")
print(f"Model parameters: {model.num_parameters():,}")


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Model loaded!
Model parameters: 241,734,912


In [ ]:
def preprocess(batch):
    audio = batch["audio"]

    batch["input_features"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt"
    ).input_features[0]

    batch["labels"] = processor.tokenizer(
        batch["transcription"]
    ).input_ids

    return batch

dataset = dataset.map(preprocess, remove_columns=dataset.column_names)
print(dataset)

Dataset({
    features: ['input_features', 'labels'],
    num_rows: 2120
})


In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
  processor: Any

  def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:   ## call is inbuilt method , helps in writing a class with the instance of a function
        input_features = [{"input_features": f["input_features"]} for f in features]     ## This is a list comprehension.
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")   ## padding so that they have same dimensions

        label_features = [{"input_ids": f["labels"]} for f in features]      # These are the tokenized transcription sentences.
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(   #The tokenizer also creates an attention_mask
            labels_batch.attention_mask.ne(1), -100
        )
#input_ids -> words converted into tokens and labels -> correct ans the model is supposed to predict
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("Data collator ready!")

Data collator ready!


In [ ]:
train_dataset = load_dataset("google/fleurs", "hi_in", split="train")
eval_dataset = load_dataset("google/fleurs", "hi_in", split="validation")

train_dataset = train_dataset.map(preprocess, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(preprocess, remove_columns=eval_dataset.column_names)

print(f"Train samples: {len(train_dataset)}")
print(f"Eval samples: {len(eval_dataset)}")

Train samples: 2120
Eval samples: 239


In [ ]:
import evaluate  #evaluate is a hugging face library which contains all the metrics like wer, bleu etc

metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions #These are the model's predictions.
    label_ids = pred.label_ids  #These are the correct answers.

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True) #this part converts tokens back into the text for predicted ans
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True) #does the same but for correct ans

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

print("Metric ready!")

Metric ready!


In [ ]:
# now  we are going to set up the training arguements telling our model how to train

from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-hindi",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # tricks the model into thinking that batch size is 16 but we are giving 8 to the gpu
    learning_rate=1e-5,    #determines the size of the steps an AI model takes to adjust its parameters (weights) during training
    num_train_epochs=3,     #1 epoch means the whole dataset has been seen once , we are training the model thrice on the same dataset
    eval_strategy="epoch",   #check the performace of model after every epoch
    save_strategy="epoch",  #save checkppints after every epoch
    logging_strategy="epoch", #print loss, learning rate and epoch
         #after every 25 steps print loss, learning rate and epoch
    predict_with_generate=True,
    fp16=True,    # using 16 bit numbers
    load_best_model_at_end=True, #load best model based on which has lowest WER
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)

print("Training arguments ready!")




Training arguments ready!


In [ ]:

from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics,

)

print("Trainer ready!")
trainer.train()

Trainer ready!


Epoch,Training Loss,Validation Loss,Wer
1,0.994869,0.286547,32.067039
2,0.498377,0.281578,29.981378
3,0.318309,0.288739,30.037244


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its para

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=399, training_loss=0.6038517294671004, metrics={'train_runtime': 2405.3703, 'train_samples_per_second': 2.644, 'train_steps_per_second': 0.166, 'total_flos': 1.8354031460352e+18, 'train_loss': 0.6038517294671004, 'epoch': 3.0})

In [ ]:
#now we move to the inference

import torch

sample = eval_dataset[0]

#covert input features into tensor and move to gpu
input_features = torch.tensor(sample["input_features"]).unsqueeze(0).to("cuda") #unsqueeze adds a new dimension bcoz tensors are 3d

#generating transcription
with torch.no_grad():
  predicted_ids = model.generate(input_features,     #.generate() generates text token by token (audio->encoder->decoder-> ......)
                                 language = "hi",
                                 task = "transcribe")

#decode the token ids back to text
transcription = processor.tokenizer.batch_decode(
    predicted_ids,
    skip_special_tokens = True
)[0]

print("transcription:", transcription)

transcription: मेज़र्किन खाना भूमध्य सागर में सामान क्षेत्रों की तरह रोटी सबजीयों और मास विशेष रूप से सुअर का मास पर आधारित है और जैतून के तेलका उपयोग करता है


In [ ]:
import torch

# take one sample from eval dataset
sample = eval_dataset[0]

# get the correct transcription (ground truth)
correct_transcription = processor.tokenizer.decode(
    sample["labels"],
    skip_special_tokens=True
)

# convert input features to tensor and move to GPU
input_features = torch.tensor(sample["input_features"]).unsqueeze(0).to("cuda")

# generate transcription
with torch.no_grad():
    predicted_ids = model.generate(
        input_features,
        language="hi",
        task="transcribe"
    )

# decode predicted tokens to text
predicted_transcription = processor.tokenizer.batch_decode(
    predicted_ids,
    skip_special_tokens=True
)[0]

# calculate WER for this single sample
sample_wer = metric.compute(
    predictions=[predicted_transcription],   #this .compute() expects a list[str]
    references=[correct_transcription]
)

print("Correct    :", correct_transcription)
print("Predicted  :", predicted_transcription)
print(f"WER        : {sample_wer * 100:.2f}%")

Correct    : मेजरकेन खाना भूमध्य सागर में समान क्षेत्रों की तरह रोटी सब्जियों और मांस विशेष रूप से सूअर का मांस पर आधारित है और जैतून के तेल का उपयोग करता है।
Predicted  : मेज़र्किन खाना भूमध्य सागर में सामान क्षेत्रों की तरह रोटी सबजीयों और मास विशेष रूप से सुअर का मास पर आधारित है और जैतून के तेलका उपयोग करता है
WER        : 30.00%


In [ ]:
# save model and processor to hugging face hub
model.push_to_hub("whisper-small-hindi-finetuned")
processor.push_to_hub("whisper-small-hindi-finetuned")

print("Model saved to Hugging Face!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...fmjqvxa/model.safetensors:   0%|          |  572kB /  967MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Model saved to Hugging Face!


In [ ]:
my_model = WhisperForConditionalGeneration.from_pretrained("shady1628/whisper-small-hindi-finetuned")
my_processor = WhisperProcessor.from_pretrained("shady1628/whisper-small-hindi-finetuned")

my_model = my_model.to("cuda")

print("My model loaded from Hub!")

config.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/4.60k [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.11k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.93M [00:00<?, ?B/s]

My model loaded from Hub!
